# Tongue ROM Landmark Visualization

This notebook allows you to select a subject and a video from the `data/` folder, and visualize the landmarks overlaid on the video frames.

**Note:** If you see rendering errors, try restarting the kernel and clearing all outputs.

In [8]:
import os
import json
import cv2
import numpy as np
from pathlib import Path
from IPython.display import display, clear_output
import ipywidgets as widgets
import io

# Setup Paths
ROOT_DIR = Path('..')
DATA_DIR = ROOT_DIR / 'data'
if not DATA_DIR.exists():
    DATA_DIR = Path('data')

print(f"Using DATA_DIR: {DATA_DIR.absolute()}")

# Helper to convert CV2 frame to Widget format
def frame_to_widget(frame):
    _, buffer = cv2.imencode('.jpg', cv2.cvtColor(frame, cv2.COLOR_RGB2BGR))
    return buffer.tobytes()

Using DATA_DIR: /home/lucas_takanori/oro_lat_el/poc/../data


In [9]:
def get_subjects():
    if not DATA_DIR.exists(): return []
    return sorted([p.name for p in DATA_DIR.iterdir() if p.is_dir()])

def get_videos(subject):
    subject_dir = DATA_DIR / subject
    return sorted([p.name for p in subject_dir.glob('*.webm')])

# UI Elements
subject_dropdown = widgets.Dropdown(options=get_subjects(), description='Subject:')
video_dropdown = widgets.Dropdown(description='Video:')
crop_toggle = widgets.Checkbox(value=False, description='Show Mouth ROI')
video_display = widgets.Image(format='jpeg', width=800)
slider = widgets.IntSlider(description='Frame:', continuous_update=True)
play_button = widgets.Play(interval=100, step=1, description="Press play")
status_label = widgets.Label(value="Select a video to start")

# Data state
state = {'frames': [], 'landmarks': [], 'h': 0, 'w': 0}

def update_video_list(*args):
    if subject_dropdown.value:
        videos = get_videos(subject_dropdown.value)
        video_dropdown.options = videos
    else:
        video_dropdown.options = []

def load_video_data(*args):
    video_filename = video_dropdown.value
    if not video_filename:
        status_label.value = "No video selected."
        return
    
    status_label.value = f"Loading {video_filename}..."
    video_path = DATA_DIR / subject_dropdown.value / video_filename
    landmarks_path = DATA_DIR / subject_dropdown.value / video_filename.replace('.webm', '_landmarks.json')
    
    # Load Landmarks
    state['landmarks'] = []
    if landmarks_path.exists():
        try:
            with open(landmarks_path, 'r') as f:
                state['landmarks'] = json.load(f).get('landmarks', [])
        except Exception as e:
            print(f"Error loading landmarks: {e}")
    
    # Load Video
    cap = cv2.VideoCapture(str(video_path))
    state['frames'] = []
    while True:
        ret, frame = cap.read()
        if not ret: break
        state['frames'].append(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB))
    cap.release()
    
    if state['frames']:
        state['h'], state['w'], _ = state['frames'][0].shape
        slider.max = len(state['frames']) - 1
        slider.value = 0
        play_button.max = slider.max
        status_label.value = f"Ready: {len(state['frames'])} frames"
        render_frame()
    else:
        status_label.value = "Error loading video"

def render_frame(*args):
    idx = slider.value
    if not state['frames'] or idx >= len(state['frames']): return
    
    # Copy original frame to draw on
    img = state['frames'][idx].copy()
    h, w = state['h'], state['w']
    
    if idx < len(state['landmarks']):
        lm = state['landmarks'][idx]['lm']
        
        # 1. Draw all landmarks (subtle dots)
        for p in lm:
            cv2.circle(img, (int(p['x']*w), int(p['y']*h)), 1, (0, 255, 255), -1)
            
        # 2. Draw Lip Contours
        outer_lip = [61, 146, 91, 181, 84, 17, 314, 405, 321, 375, 291, 308, 324, 318, 402, 317, 14, 87, 178, 88, 95, 78]
        pts = np.array([[lm[i]['x']*w, lm[i]['y']*h] for i in outer_lip if i < len(lm)], np.int32)
        if len(pts) > 0:
            cv2.polylines(img, [pts], True, (255, 0, 0), 2)
        
        # 3. Draw ROI Box (what build_dataset.py uses)
        if crop_toggle.value:
            roi_idxs = [61, 291, 78, 308, 13, 14, 0]
            valid_idxs = [i for i in roi_idxs if i < len(lm)]
            if valid_idxs:
                xs = [lm[i]['x'] * w for i in valid_idxs]
                ys = [lm[i]['y'] * h for i in valid_idxs]
                cx, cy = (min(xs) + max(xs)) / 2, (min(ys) + max(ys)) / 2
                side = max(max(xs)-min(xs), max(ys)-min(ys)) * 2.2
                x0, y0 = int(cx - side/2), int(cy - side/2)
                x1, y1 = int(cx + side/2), int(cy + side/2)
                cv2.rectangle(img, (x0, y0), (x1, y1), (0, 255, 0), 2)

    video_display.value = frame_to_widget(img)

# Wiring
subject_dropdown.observe(update_video_list, 'value')
video_dropdown.observe(load_video_data, 'value')
slider.observe(render_frame, 'value')
crop_toggle.observe(render_frame, 'value')
widgets.jslink((play_button, 'value'), (slider, 'value'))

# Layout
controls = widgets.VBox([
    widgets.HBox([subject_dropdown, video_dropdown]),
    widgets.HBox([play_button, slider, crop_toggle]),
    status_label
])
display(controls, video_display)

# Init
update_video_list()
if subject_dropdown.value:
    load_video_data()

Image(value=b'', format='jpeg', width='800')